# Dropout — A Simple Way to Prevent Neural Networks from Overfitting (2014)

_Papers · Foundations & Optimization_

**Randomly switch off a fraction of units each training step, so the network can't rely on any single unit and overfits less.**

---

This notebook builds dropout from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We implement **inverted dropout** from scratch and check it against PyTorch's `nn.Dropout`. We build it in four steps: (1) the dropout function, (2) checking it in eval and train mode, (3) the paper's worked example, (4) dropping it into a small network.

### Step 1 — Write the inverted-dropout function

Dropout randomly zeros out units during training. We use the **inverted** form: we drop each unit with probability `p_drop`, then scale the survivors by `1/keep` so the expected value of every unit is unchanged. That scaling is what lets us do *nothing* at eval time — the function just returns its input when `training=False`.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def my_dropout(x, p_drop, training):
    """Inverted dropout from scratch — Section 4 of Srivastava et al. (2014).
    p_drop = probability of DROPPING a unit (same convention as nn.Dropout(p))."""
    if not training or p_drop == 0.0:
        return x                                  # EVAL: identity, dropout skipped
    keep = 1.0 - p_drop                           # retention probability p
    mask = (torch.rand_like(x) < keep).float()    # r ~ Bernoulli(keep)
    return x * mask / keep                        # zero dropped + scale kept by 1/p

### Step 2 — Check it against `nn.Dropout` in both modes

Two sanity checks. **Oracle 1:** in eval mode dropout is the identity, so our function must exactly match `nn.Dropout` set to `.eval()`. **Oracle 2:** in train mode the mask is random, but averaged over many masks the output should preserve the input's expected value — and match PyTorch's own averaged output. We feed a large constant tensor so the column means are easy to read.

In [ ]:
# ---- ORACLE 1: eval mode is exactly the identity, same as nn.Dropout in eval ----
p = 0.5
x = torch.randn(8, 16)
ref = nn.Dropout(p)
ref.eval()
ok_eval = torch.allclose(my_dropout(x, p, training=False), ref(x))
print("allclose eval (both identity):", ok_eval)          # expect True

# ---- ORACLE 2: train mode preserves the expected value (over many masks) ----
torch.manual_seed(1)
big = torch.full((100000, 8), 3.0)                 # constant input, easy to average
mine_mean = my_dropout(big, p, training=True).mean(0)   # ~ 3.0 for every column
ref.train()
ref_mean = ref(big).mean(0)                        # PyTorch also preserves E[.]
print("my train E[.]   ~", mine_mean[:4].round(decimals=3).tolist())   # ~ [3,3,3,3]
print("nn train E[.]   ~", ref_mean[:4].round(decimals=3).tolist())    # ~ [3,3,3,3]
print("expected-value preserved:",
      torch.allclose(mine_mean, torch.full((8,), 3.0), atol=0.05),
      "| matches nn.Dropout:",
      torch.allclose(mine_mean, ref_mean, atol=0.05))

### Step 3 — Reproduce the worked example

Take the vector `y = [2, 4, 6, 8]` with `p_drop = 0.5` and a mask that keeps units 1, 3, 4 and drops unit 2. The train output zeros the dropped unit and doubles the survivors (`1/keep = 2`). Averaging the random output over 200k masks recovers the original `y`, confirming the expected value is preserved.

In [ ]:
# ---- recompute the worked example: y=[2,4,6,8], p_drop=0.5, mask keeps [1,0,1,1] ----
y = torch.tensor([2., 4., 6., 8.])
mask = torch.tensor([1., 0., 1., 1.])
keep = 0.5
print("worked-example train out:", (y * mask / keep).tolist())   # [4, 0, 12, 16]

# empirical mean over 200k masks ~ original y
torch.manual_seed(2)
samp = torch.stack([my_dropout(y, 0.5, True) for _ in range(200000)]).mean(0)
print("empirical E[train out]  :", samp.round(decimals=2).tolist())  # ~ [2,4,6,8]

### Step 4 — Use it as a layer in a small network

Finally, drop the function into a 2-layer classifier exactly where you would place an `nn.Dropout` layer: right after the hidden activation. Passing `self.training` makes it honor the module's train/eval state automatically. The forward pass returns logits of shape `[batch, 2]`.

In [ ]:
# ---- drop it into a 2-layer net (use it like a layer) ----
class Net(nn.Module):
    def __init__(self, p_drop=0.5):
        super().__init__()
        self.fc1 = nn.Linear(16, 32)
        self.fc2 = nn.Linear(32, 2)
        self.p = p_drop

    def forward(self, z):
        h = torch.relu(self.fc1(z))
        h = my_dropout(h, self.p, self.training)   # dropout after the hidden activation
        return self.fc2(h)

net = Net()
net.train()
print("net output shape:", net(torch.randn(5, 16)).shape)  # torch.Size([5, 2])

## Visualize the data & results

_Train the same small 2-hidden-layer classifier on the same small, noisy task — once with no dropout, once with dropout. Does dropout reduce overfitting (a smaller gap between training and test loss)?_

### Step 1 — Make a noisy 2D dataset and helpers

We build a small classification task: points inside a circle of radius 1.3 are one class, outside the other, with **10% of labels flipped** as noise. Noise is what lets a high-capacity net overfit — it can memorize the flipped points. We also define the sigmoid and the cross-entropy loss we'll measure on train and test.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    X = r.normal(0, 1, (n, 2))
    rad = np.sqrt((X**2).sum(1))
    ytrue = (rad > 1.3).astype(float)             # circular decision boundary
    flip = r.random(n) < 0.10                     # flip 10% of the labels: noise
    y = np.where(flip, 1 - ytrue, ytrue)
    return X.astype(np.float64), y

Xtr, ytr = make(200, 1)                           # small training set (overfits easily)
Xte, yte = make(4000, 2)                          # large test set

def sig(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def ce(P, X, y):
    W1, b1, W2, b2, W3, b3 = P
    a1 = np.maximum(X @ W1 + b1, 0)
    a2 = np.maximum(a1 @ W2 + b2, 0)
    o = np.clip(sig(a2 @ W3 + b3).ravel(), 1e-7, 1 - 1e-7)
    return float(-np.mean(y * np.log(o) + (1 - y) * np.log(1 - o)))

### Step 2 — Train a 3-layer net with optional dropout

This is a hand-written forward and backward pass for a 3-layer MLP. When `p_drop > 0` we apply inverted dropout to both hidden activations on the forward pass, and — crucially — multiply by the **same mask and `1/keep` scale** on the backward pass so the gradients match the masked forward. When `p_drop == 0` dropout is skipped entirely. The function returns the final train and test cross-entropy.

In [ ]:
def train(p_drop, epochs=300, lr=0.05, h=256, bs=32, seed=2):
    r = np.random.default_rng(seed)
    W1 = r.normal(0, 0.3, (2, h)); b1 = np.zeros(h)
    W2 = r.normal(0, 0.1, (h, h)); b2 = np.zeros(h)
    W3 = r.normal(0, 0.1, (h, 1)); b3 = np.zeros(1)
    keep = 1 - p_drop
    n = len(ytr)
    for ep in range(epochs):
        idx = r.permutation(n)
        for s in range(0, n, bs):
            b = idx[s:s + bs]
            Xb = Xtr[b]; yb = ytr[b]; m = len(b)
            # ---- forward ----
            z1 = Xb @ W1 + b1; a1 = np.maximum(z1, 0)
            if p_drop > 0:
                m1 = (r.random(a1.shape) > p_drop).astype(float)   # keep mask, layer 1
                a1d = a1 * m1 / keep                                # inverted dropout
            else:
                a1d = a1
            z2 = a1d @ W2 + b2; a2 = np.maximum(z2, 0)
            if p_drop > 0:
                m2 = (r.random(a2.shape) > p_drop).astype(float)   # keep mask, layer 2
                a2d = a2 * m2 / keep
            else:
                a2d = a2
            out = np.clip(sig(a2d @ W3 + b3).ravel(), 1e-7, 1 - 1e-7)
            # ---- backward ----
            d3 = (out - yb).reshape(-1, 1) / m
            dW3 = a2d.T @ d3; db3 = d3.sum(0); da2 = d3 @ W3.T
            if p_drop > 0:
                da2 = da2 * m2 / keep                               # same mask on the gradient
            dz2 = da2 * (z2 > 0)
            dW2 = a1d.T @ dz2; db2_ = dz2.sum(0); da1 = dz2 @ W2.T
            if p_drop > 0:
                da1 = da1 * m1 / keep
            dz1 = da1 * (z1 > 0)
            dW1 = Xb.T @ dz1; db1_ = dz1.sum(0)
            # ---- SGD update ----
            W1 -= lr * dW1; b1 -= lr * db1_
            W2 -= lr * dW2; b2 -= lr * db2_
            W3 -= lr * dW3; b3 -= lr * db3
    P = (W1, b1, W2, b2, W3, b3)
    return ce(P, Xtr, ytr), ce(P, Xte, yte)

### Step 3 — Compare overfitting with and without dropout

Now run the same training twice — once with `p_drop = 0.0` and once with `p_drop = 0.3` — and print the train loss, test loss, and the **gap** between them. The gap is the real overfitting signal: dropout should raise the train loss slightly (it's harder to fit) but lower the test loss, shrinking the gap.

In [ ]:
for pd in [0.0, 0.3]:
    tr, te = train(pd)
    print(f"p_drop={pd}: train {tr:.3f}  test {te:.3f}  gap {te-tr:.3f}")
# p_drop=0.0: train 0.228  test 0.560  gap 0.332  (overfits)
# p_drop=0.3: train 0.265  test 0.544  gap 0.279  (less overfitting)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Apply inverted dropout with drop probability 0.2 (so keep $p=0.8$, scale $1/p=1.25$) to the vector $[5,10,15]$, with a mask that keeps units 1 and 3 and drops unit 2. What is the train output, and what is the eval output?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mask $\mathbf{r}=[1,0,1]$; multiply: $[5,0,15]$. — _Dropped unit's output becomes 0._
- Scale kept units by $1/p=1.25$: $[5\times1.25,\,0,\,15\times1.25]=[6.25,\,0,\,18.75]$. — _Inverted-dropout $1/p$ scaling keeps the expected value unchanged._
- Eval mode: return the input unchanged, $[5,10,15]$. — _Dropout is skipped at test time._

**Answer:** Train output $=[6.25,\,0,\,18.75]$; eval output $=[5,10,15]$. Check the expectation of unit 1: $0.8\times(5\times1.25)+0.2\times 0 = 5$, the original value.

</details>

**Problem 2.** Ablation: in the CODE's small net, set the dropout rate to 0 (turn dropout off) and train on the same noisy data. What do you expect to happen to the training loss and the gap between training and test loss?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $p_{\text{drop}}=0$ so no units are dropped and no scaling is applied. — _This removes the regularizer; the net trains at full capacity._
- Train and watch both training and test loss. — _Dropout's effect shows up in the gap, not the training number alone._
- Compare to the dropout run (the CODEVIZ chart). — _Same data, same architecture &mdash; only dropout differs._

**Answer:** Without dropout the net memorizes more: in our small run (CODEVIZ) the training loss falls lower (~0.23 vs ~0.27 with dropout), but the test loss ends higher (~0.56 vs ~0.54) and its curve rises after an early dip &mdash; the train&ndash;test gap grows from ~0.28 (dropout) to ~0.33 (no dropout). This reproduces the paper's qualitative claim that dropout reduces overfitting. These are our small-run numbers, not the paper's.

</details>

**Problem 3.** The paper's original dropout keeps activations unscaled at train and scales weights by $p$ at test ($W_{test}=pW$). Inverted dropout scales by $1/p$ at train and does nothing at test. Show the two give the same expected activation in both modes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Original, train: kept unit outputs $y$ with prob $p$, else $0$, so $\mathbb{E}=py$. Test: weight scaled by $p$, unit always on, so output contribution $\propto p\cdot y$. — _Train and test expectations both equal $py$ &mdash; consistent, but train activations are $p$-shrunk._
- Inverted, train: kept unit outputs $y/p$ with prob $p$, else $0$, so $\mathbb{E}=p\cdot(y/p)=y$. Test: unit always on, weight unscaled, output $y$. — _Train and test expectations both equal $y$._

**Answer:** Both schemes make the expected train activation equal the test activation; they differ only by an overall constant factor ($py$ vs $y$) that the next layer's weights absorb during training. Inverted dropout is preferred because the test path is an ordinary unscaled network &mdash; nothing to remember to rescale at inference.

</details>